# CNN Model Training & Evaluation
## Module 3 - Computer Vision Learning

**⚠️ Run this notebook on Google Colab with T4 GPU enabled**

This notebook handles:
1. Build CNN model
2. Train with early stopping and checkpoints
3. Evaluate model
4. Plot loss and accuracy
5. Test with real data

## Setup & Imports

In [ ]:
# Check GPU availability
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

if not tf.config.list_physical_devices('GPU'):
    print("\n⚠️ WARNING: No GPU found! Please enable GPU in Colab:")
    print("   Runtime → Change runtime type → GPU (T4 recommended)")

In [ ]:
# Mount Google Drive (only for Colab)
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

In [ ]:
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# TensorFlow & Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("✓ All imports successful")

In [ ]:
# Define paths - adjust if dataset is in Google Drive
# Example for Google Drive:
DATASET_BASE = Path('/content/drive/MyDrive/Colab/datasets/flowers')  # Change this path
DATASET_SPLITS = DATASET_BASE / 'splits'
TRAIN_DIR = DATASET_SPLITS / 'train'
VAL_DIR = DATASET_SPLITS / 'val'
TEST_DIR = DATASET_SPLITS / 'test'

# Create output directory
OUTPUT_DIR = Path('/content/drive/MyDrive/Colab/flower_classifier_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Train directory: {TRAIN_DIR}")
print(f"Train directory exists: {TRAIN_DIR.exists()}")
print(f"Output directory: {OUTPUT_DIR}")

## Step 1: Data Loading & Preprocessing

In [ ]:
# Image dimensions and batch size
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
NUM_CLASSES = 3

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# No augmentation for validation and test
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("Data augmentation configured")

In [ ]:
# Load data generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Get class names
class_names = list(train_generator.class_indices.keys())
class_names.sort()

print(f"\nClass names: {class_names}")
print(f"Train batches: {len(train_generator)}")
print(f"Val batches: {len(val_generator)}")
print(f"Test batches: {len(test_generator)}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()

# Reset generator
train_generator.reset()
images, labels = next(train_generator)

for i in range(9):
    axes[i].imshow(images[i])
    class_idx = np.argmax(labels[i])
    axes[i].set_title(class_names[class_idx], fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Sample images visualization saved")

## Step 2: Build CNN Model

In [ ]:
def build_cnn_model(input_shape, num_classes):
    """
    Build a CNN model from scratch.
    """
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 4
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Flatten and Dense layers
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Build model
model = build_cnn_model((IMG_HEIGHT, IMG_WIDTH, 3), NUM_CLASSES)

# Display model architecture
print("\n" + "="*60)
print("MODEL ARCHITECTURE")
print("="*60)
model.summary()

print(f"\nTotal parameters: {model.count_params():,}")

In [ ]:
# Compile model
optimizer = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn = keras.losses.CategoricalCrossentropy()
metrics = ['accuracy']

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=metrics
)

print("✓ Model compiled successfully")

## Step 3: Train Model with Early Stopping & Checkpoints

In [ ]:
# Create callbacks
checkpoint_dir = OUTPUT_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Early stopping
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

# Model checkpoint
model_checkpoint = callbacks.ModelCheckpoint(
    filepath=str(checkpoint_dir / 'best_model.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Learning rate reduction
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Tensorboard
log_dir = OUTPUT_DIR / 'logs'
tensorboard = callbacks.TensorBoard(
    log_dir=str(log_dir),
    histogram_freq=1,
    write_graph=True
)

callbacks_list = [early_stopping, model_checkpoint, reduce_lr, tensorboard]

print(f"Checkpoint directory: {checkpoint_dir}")
print("✓ Callbacks configured")

In [ ]:
# Train model
print("\n" + "="*60)
print("STARTING MODEL TRAINING")
print("="*60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

EPOCHS = 100

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1
)

print(f"\nEnd time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✓ Training completed")

## Step 4: Plot Loss & Accuracy

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Training history plots saved")

In [ ]:
# Print final metrics
print("\n" + "="*60)
print("TRAINING METRICS")
print("="*60)

final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print(f"\nFinal Epoch Metrics:")
print(f"  Train Accuracy: {final_train_acc:.4f}")
print(f"  Val Accuracy:   {final_val_acc:.4f}")
print(f"  Train Loss:     {final_train_loss:.4f}")
print(f"  Val Loss:       {final_val_loss:.4f}")

best_val_acc = max(history.history['val_accuracy'])
best_epoch = history.history['val_accuracy'].index(best_val_acc) + 1

print(f"\nBest Val Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")

## Step 5: Evaluate Model on Test Set

In [ ]:
# Load best model
best_model = keras.models.load_model(checkpoint_dir / 'best_model.h5')
print("✓ Best model loaded from checkpoint")

In [ ]:
# Evaluate on test set
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

test_loss, test_accuracy = best_model.evaluate(test_generator, verbose=0)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Get predictions on test set
test_generator.reset()
predictions = best_model.predict(test_generator, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes

print("✓ Predictions generated")

In [ ]:
# Classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)

report = classification_report(
    true_classes,
    predicted_classes,
    target_names=class_names,
    digits=4
)

print(report)

In [ ]:
# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
ax.set_title('Confusion Matrix - Test Set', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix saved")

## Step 6: Test with Real Data

In [ ]:
from PIL import Image

def predict_single_image(model, image_path, class_names):
    """
    Predict class for a single image.
    """
    # Load and preprocess image
    img = Image.open(image_path).convert('RGB')
    img = img.resize((IMG_HEIGHT, IMG_WIDTH))
    img_array = np.array(img) / 255.0
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Predict
    prediction = model.predict(img_batch, verbose=0)
    predicted_class = np.argmax(prediction[0])
    confidence = prediction[0][predicted_class]
    
    return class_names[predicted_class], confidence, prediction[0]

print("✓ Prediction function ready")

In [ ]:
# Test on random samples from test set
test_sample_dir = TEST_DIR
all_test_images = []

for class_dir in test_sample_dir.iterdir():
    if class_dir.is_dir():
        for img_file in class_dir.glob('*.jpg'):
            all_test_images.append((img_file, class_dir.name))
        for img_file in class_dir.glob('*.jpeg'):
            all_test_images.append((img_file, class_dir.name))
        for img_file in class_dir.glob('*.png'):
            all_test_images.append((img_file, class_dir.name))

# Select random samples
sample_indices = np.random.choice(len(all_test_images), size=min(9, len(all_test_images)), replace=False)
sample_images = [all_test_images[i] for i in sample_indices]

print(f"\n" + "="*60)
print("TEST PREDICTIONS ON RANDOM SAMPLES")
print("="*60)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.ravel()

for idx, (img_path, true_class) in enumerate(sample_images):
    # Predict
    pred_class, confidence, all_probs = predict_single_image(best_model, img_path, class_names)
    
    # Load and display image
    img = Image.open(img_path)
    axes[idx].imshow(img)
    
    # Title with prediction and true label
    is_correct = pred_class == true_class
    symbol = '✓' if is_correct else '✗'
    title = f"{symbol} Pred: {pred_class}\nTrue: {true_class}\nConf: {confidence:.2%}"
    color = 'green' if is_correct else 'red'
    axes[idx].set_title(title, fontweight='bold', color=color)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Test predictions visualization saved")

In [ ]:
# Interactive prediction - test specific images
def test_image_interactive(image_path_str):
    """
    Test prediction on a specific image.
    Provide full path to image.
    """
    image_path = Path(image_path_str)
    
    if not image_path.exists():
        print(f"Error: File not found - {image_path}")
        return
    
    pred_class, confidence, all_probs = predict_single_image(best_model, image_path, class_names)
    
    print(f"\n{'='*50}")
    print(f"Image: {image_path.name}")
    print(f"{'='*50}")
    print(f"\nPredicted Class: {pred_class}")
    print(f"Confidence: {confidence:.2%}")
    print(f"\nClass Probabilities:")
    for class_name, prob in zip(class_names, all_probs):
        bar = '█' * int(prob * 50)
        print(f"  {class_name:12s}: {prob:.4f} {bar}")
    
    # Display image
    img = Image.open(image_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"Predicted: {pred_class} ({confidence:.2%})", fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Example usage:
# test_image_interactive('/path/to/your/test/image.jpg')

print("\n✓ Interactive testing function ready")
print("  Usage: test_image_interactive('/path/to/image.jpg')")

## Save Model & Summary

In [ ]:
# Save final model
model_path = OUTPUT_DIR / 'flower_classifier_model.h5'
best_model.save(str(model_path))
print(f"✓ Model saved: {model_path}")

# Save model as SavedModel format (recommended for production)
savedmodel_path = OUTPUT_DIR / 'flower_classifier_savedmodel'
best_model.save(str(savedmodel_path))
print(f"✓ SavedModel saved: {savedmodel_path}")

In [ ]:
# Create summary report
summary_report = f"""
FLOWER CLASSIFICATION MODEL - TRAINING SUMMARY
{'='*60}

MODEL ARCHITECTURE:
  Type: CNN (Convolutional Neural Network)
  Total Parameters: {model.count_params():,}
  Input Shape: {IMG_HEIGHT}x{IMG_WIDTH}x3
  Classes: {NUM_CLASSES} ({', '.join(class_names)})

TRAINING CONFIGURATION:
  Epochs: {len(history.history['loss'])}
  Batch Size: {BATCH_SIZE}
  Optimizer: Adam (lr=1e-3)
  Loss Function: Categorical Crossentropy
  Data Augmentation: Yes

TRAINING RESULTS:
  Best Val Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})
  Final Train Accuracy: {final_train_acc:.4f}
  Final Val Accuracy: {final_val_acc:.4f}
  Final Train Loss: {final_train_loss:.4f}
  Final Val Loss: {final_val_loss:.4f}

TEST SET PERFORMANCE:
  Test Accuracy: {test_accuracy:.4f}
  Test Loss: {test_loss:.4f}

OUTPUTS:
  ✓ Model: flower_classifier_model.h5
  ✓ SavedModel: flower_classifier_savedmodel/
  ✓ Training History: training_history.png
  ✓ Confusion Matrix: confusion_matrix.png
  ✓ Test Predictions: test_predictions.png
  ✓ Sample Images: sample_images.png

GENERATED: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

print(summary_report)

# Save report
with open(OUTPUT_DIR / 'training_summary.txt', 'w') as f:
    f.write(summary_report)

print(f"\n✓ Summary saved: {OUTPUT_DIR / 'training_summary.txt'}")

In [ ]:
print("\n" + "="*60)
print("✓ TRAINING AND EVALUATION COMPLETE")
print("="*60)
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(f"\nYou can now:")
print(f"  1. Download the model for deployment")
print(f"  2. Review the training history and metrics")
print(f"  3. Use test_image_interactive() to test on custom images")